In [1]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langchain_ollama import ChatOllama

# Initialize the local Ollama instance
model = ChatOllama(
    model="llama3.2:3b", 
    temperature=0
)

In [2]:
# Create baseline agent
agent = create_agent(model=model)

question = HumanMessage(content="Quelle est la capitale de la lune?")
response = agent.invoke({"messages": [question]})

print(response['messages'][-1].content)

La Lune n'a pas de capitale. La Lune est un satellite naturel de la Terre et ne possède aucune structure humaine ou artificielle qui puisse être considérée comme une capitale.

Il convient de noter que la Lune a été habitée par des êtres vivants dans certaines œuvres littéraires, telles que dans le roman "La Lune est un piano d'Alberto Moravia" ou dans les films "La Lune" (1972) et "Moon" (2009). Cependant, ces représentations sont purement fictives et ne reflètent pas la réalité scientifique ou géologique de la Lune.

En résumé, il n'y a pas de capitale sur la Lune.


## Agent avec System Message: un message de contrôle qui définit le comportement global du modèle.

In [3]:
system_prompt = "Vous êtes un auteur de science-fiction; créez une capitale à la demande des utilisateurs."

scifi_agent = create_agent(
    model=model,
    system_prompt=system_prompt
)

response = scifi_agent.invoke({"messages": [question]})
print(response['messages'][-1].content)

La question classique !

Malheureusement, il n'y a pas de capitale sur la Lune. La Lune est un satellite naturel de la Terre et n'a pas d'atmosphère ou de vie organisée. Il n'y a pas non plus de gouvernement ou de population permanente sur la Lune.

Cependant, il y a eu des missions humaines sur la Lune, notamment lors du programme Apollo américain dans les années 1960 et 1970. Les astronautes qui ont visité la Lune ont déposé des drapeaux et des symboles de leur pays, mais il n'y a pas de structure permanente ou de capitale sur la Lune.

Si vous souhaitez créer une capitale sur la Lune, je serais ravi de vous aider à imaginer un monde futuriste ! Quelle serait la fonction de cette capitale ? Serait-elle un centre de recherche scientifique, un port pour les missions spatiales ou quelque chose d'autre ?


## Agent avec Few-shot learning: une méthode où le modèle apprend une nouvelle tâche ou classe à partir de quelques exemples seulement.

In [4]:
system_prompt = """
Vous êtes un auteur de science-fiction et vous devez créer une capitale spatiale à la demande d'un utilisateur.

Utilisateur: Quelle est la capitale de Mars ?
Auteur: Marsialis

Utilisateur: Quelle est la capitale de Vénus ?
Auteur: Venusovia
"""

scifi_agent = create_agent(
    model=model,
    system_prompt=system_prompt
)

response = scifi_agent.invoke({"messages": [question]})
print(response['messages'][-1].content)

Une question qui pose un défi intéressant !

En tant qu'auteur de science-fiction, je vais créer une réponse qui correspond à mon univers fictif.

La capitale de la Lune est appelée Lunaria. Située dans le cœur de la colonie lunaire, Lunaria est une ville spatiale innovante et dynamique qui abrite les institutions gouvernementales, les centres de recherche et les quartiers résidentiels des habitants de la Lune.

Lunaria est connue pour ses rues lumineuses, ses bâtiments en forme de cratère et son atmosphère unique, qui combine les éléments de l'air et du vide spatial. La ville est un symbole de la coopération internationale et de la résilience humaine dans l'espace.

Et voici quelques informations supplémentaires sur Lunaria :

* Population : 500 000 habitants
* Superficie : 100 km²
* Langue officielle : Lunarien (une langue hybride qui combine les éléments de plusieurs langues terrestres)
* Monnaie : Le Lunarien (un système monétaire basé sur des unités de travail et de ressources spa

## Agent avec réponse structurée: une sortie organisée selon un format prédéfini, plutôt que du texte libre.

In [5]:
system_prompt = """
Vous êtes un auteur de science-fiction et vous devez créer une capitale spatiale à la demande d'un utilisateur.
Veuillez respecter la structure ci-dessous.

Nom: Nom de la capitale
Localisation: Lieu où elle est située
Ambiance: Description en 2 ou 3 mots
Économie: Principaux secteurs d'activité
"""

scifi_agent = create_agent(
    model=model,
    system_prompt=system_prompt
)

response = scifi_agent.invoke({"messages": [question]})
print(response['messages'][-1].content)

Je crée une capitale spatiale pour vous !

**Nom :** Lunaria
**Localisation :** Surface de la lune, dans le bassin du lac Serenity
**Ambiance :** Futuriste et étoilée

**Économie :**

* **Tourisme spatial** : Attracter des touristes qui souhaitent découvrir les merveilles de la lune.
* **Minierie lunaire** : Extraire des ressources rares et précieuses présentes dans le sol lunaire, telles que l'uranium et les minéraux radioactifs.
* **Recherche scientifique** : Abriter des laboratoires et des instituts de recherche qui étudient la physique, la biologie et la géologie de la lune.

Lunaria est une ville spatiale dynamique et innovante, où la technologie et la science sont en constante évolution. Elle offre un avenir prometteur pour les humains qui cherchent à explorer et à comprendre notre satellite naturel.


## Agent avec réponse structurée en utilisant BaseModel: rendre la réponse facile à exploiter automatiquement par un programme ou un système.

In [6]:
from pydantic import BaseModel

class CapitalInfo(BaseModel):
    nom: str
    Localisation: str
    Ambiance: str
    Économie: str

agent = create_agent(
    model=model,
    system_prompt=system_prompt,
    response_format=CapitalInfo
)

question = HumanMessage(content="Quelle est la capitale de la lune?")
response = agent.invoke({"messages": [question]})

response["structured_response"]

CapitalInfo(nom='Lune', Localisation='Lune', Ambiance='Hostile', Économie='')

## Les tools (outils): une fonction externe que le modèle peut appeler pour effectuer une action précise qu'il ne peut pas faire seul.
### Création du Tool

In [7]:
from langchain.tools import tool

@tool("meteo_capitale")
def meteo_capitale(ville: str) -> str:
    """Donne la météo d'une capitale (valeurs fixes pour test).
    
    Args:
        ville: nom de la capitale
    """
    print("tool meteo_capitale utilisé")
    temperature = 25
    humidite = 60
    pression = 1013
    return (
        f"Météo à {ville} : "
        f"Température = {temperature}°C, "
        f"Humidité = {humidite}%, "
        f"Pression = {pression} hPa"
    )

### Ajout du Tool à l'agent

In [8]:
system_prompt = "Utilises les tools pour répondre aux questions"

agent = create_agent(
    model=model,
    tools=[meteo_capitale],
    system_prompt=system_prompt,
)

question = HumanMessage(content="Quelle est la météo à Capitole lunaire ?")
response = agent.invoke({"messages": [question]})

print(response['messages'][-1].content)

tool meteo_capitale utilisé
Voici la réponse formatée :

La météo à Capitole lunaire est actuellement :

* Température : 25°C
* Humidité : 60%
* Pression atmosphérique : 1013 hPa


## Agent sans Web Search Tool

In [9]:
agent = create_agent(model=model)

question = HumanMessage(content="Vos connaissances en matière d'apprentissage sont-elles à jour ?")
response = agent.invoke({"messages": [question]})

print(response['messages'][-1].content)

Mes connaissances sont actuellement basées sur des données jusqu'en décembre 2023, mais j'ai accès à des informations plus récentes via la recherche sur Internet. Cependant, il est important de noter que ma capacité à fournir des informations en temps réel peut être limitée par les limitations des mes sources de données et de l'actualisation régulière de mon éducation.

Ma connaissance en matière d'apprentissage couvre un large éventail de sujets, notamment :

1. Les principes de base de l'apprentissage automatique, tels que la régression linéaire, les arbres de décision et les réseaux de neurones.
2. Les algorithmes d'apprentissage supervisé, tels que le gradient descendant et les optimiseurs.
3. Les algorithmes d'apprentissage non supervisé, tels que la k-means et l'analyse en composantes principales (ACP).
4. Les techniques de traitement du langage naturel, telles que la tokenisation, la lemmatisation et la reconnaissance de modèles de langage.
5. Les applications de l'apprentissage

## Agent avec Web Search Tool
### Sortie du Tool

In [11]:
from typing import Dict, Any
from tavily import TavilyClient
from dotenv import load_dotenv

load_dotenv()
tavily_client = TavilyClient()

@tool
def web_search(query: str) -> Dict[str, Any]:
    """Effectue une recherche en ligne pour obtenir des informations actualisées."""
    return tavily_client.search(query)

web_search.invoke("Qui est le Président de commune actuel de Marrakech ?")

{'query': 'Qui est le Président de commune actuel de Marrakech ?',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://www.ville-marrakech.ma/bureau-du-conseil-communal-de-marrakech',
   'title': 'Bureau du Conseil Communal de Marrakech',
   'content': 'Mme Fatim Ezzahra EL MANSOURI. Président de la Commune de Marrakech ; M. Mohamed IDRISSI. 1er Vice président du Conseil Communal de Marrakech ; M. Abdellah',
   'score': 0.98544,
   'raw_content': None},
  {'url': 'https://fr.le360.ma/politique/presidents-de-communes-la-liste-des-limogeages-sallonge_XOWP57I3MBF5TNHHENNGCLG3VM',
   'title': "Présidents de commune: la liste des limogeages s'allonge | le360.ma",
   'content': "Dans ce cadre, deux présidents et deux vice-présidents viennent d'être révoqués de leurs fonctions dans la région de Marrakech-Safi par la",
   'score': 0.98343,
   'raw_content': None},
  {'url': 'https://fr.wikipedia.org/wiki/Liste_de_pr%C3%A9sidents_de_conseils_communaux_du_

### Sortie de l'agent

In [15]:
agent = create_agent(
    model=model,
    tools=[web_search]
)

question = HumanMessage(content="Qui est le Président de commune actuel de Marrakech ?")
response = agent.invoke({"messages": [question]})

print(response['messages'][-1].content)

{"name": "Effectue une recherche en ligne", "parameters": {"query": "Pr\u00e9sident de commune actuel de Marrakech"}}


## Agent sans mémoire

In [16]:
agent = create_agent(model=model)

question = HumanMessage(content="Bonjour, mon nom est Sami et je suis un développeur.")
response = agent.invoke({"messages": [question]})
print(response['messages'][-1].content)

question = HumanMessage(content="Quel est mon métier ?")
response = agent.invoke({"messages": [question]})
print(response['messages'][-1].content)

Bonjour Sami ! Enchanté de faire votre connaissance. Je suis ravi de discuter avec vous en tant que développeur. Qu'est-ce qui vous amène aujourd'hui ? Avez-vous besoin d'aide ou de conseils sur un projet spécifique ? Ou peut-être souhaitez-vous simplement partager vos expériences et vos connaissances avec moi ? Je suis là pour écouter et aider si je peux !
Je serais ravi de t'aider à découvrir ton métier !

Pour commencer, peux-tu me donner quelques informations sur toi-même ?

* Quels sont tes intérêts ?
* Quelles sont tes compétences ?
* As-tu déjà fait des études ou des formations ?
* Qu'est-ce que tu aimes faire dans ta vie quotidienne ?

En répondant à ces questions, nous pourrons commencer à explorer ensemble les différentes options de métiers qui pourraient correspondre à tes talents et à tes passions !

Et si tu veux, on peut également jouer un jeu pour t'aider à découvrir ton métier. Je peux te poser des questions sur toi-même, comme :

* Qu'est-ce que tu aimes faire pendant 

## Agent avec mémoire

In [17]:
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model=model,
    checkpointer=InMemorySaver(),
)

config = {"configurable": {"thread_id": "1"}}

question = HumanMessage(content="Bonjour, mon nom est Sami et je suis un développeur.")
response = agent.invoke({"messages": [question]}, config)

question = HumanMessage(content="Quel est mon métier ?")
response = agent.invoke({"messages": [question]}, config)

print(response['messages'][-1].content)

Vous êtes un développeur ! C'est un métier très varié et passionnant, car vous avez l'occasion de créer des solutions innovantes et de résoudre des problèmes complexes pour les entreprises et les individus. En tant que développeur, vous travaillez souvent avec des langages de programmation, des frameworks et des bases de données pour concevoir et développer des applications, des sites web ou des logiciels.

Qu'est-ce que vous développez actuellement ? Un site web, une application mobile, un jeu ou peut-être quelque chose d'autre ?
